# Homogeneous allpass FDN (MIMO)

Example for an allpass FDN with **homogeneous decay** so that all poles have the same decay rate. Compared to the SISO case, the MIMO has considerably more degrees of freedom.

See *Allpass Feedback Delay Networks*, Sebastian J. Schlecht (IEEE Trans. Signal Processing).

— Original MATLAB: Sebastian J. Schlecht, 10 June 2020

## Setup

In [ ]:
import numpy as np
import pyFDN

np.random.seed(1)

## Build homogeneous allpass FDN

Random delays, gain matrix **G**, random mixing matrix **U**, combined to the feedback matrix **A**. The remaining coefficients are reconstructed by completing the allpass FDN.

In [ ]:
Fs = 48000
N = 8
numio = N

delays = np.random.randint(800, 1800, size=N)  # delays in samples, 1..30
g = pyFDN.rt_to_gain_per_sample(0.6, Fs)
G = np.diag(g ** delays)  # gain matrix
U = pyFDN.random_orthogonal(N)
A = G @ U 

B, C, D, X = pyFDN.complete_fdn(A, N, numio)


## Test: uniallpass

Check that the system is uniallpass, i.e., allpass for any delays.

In [ ]:
is_a, _ = pyFDN.is_uniallpass(A, B, C, D, tol=1e-7)
assert is_a, "Expected allpass for homogeneous FDN with these delays"
print("is_allpass: OK")

## Plot system matrix

Visualize system matrix as 2×2 block heatmaps.

In [ ]:
fig = pyFDN.plot_system_matrix(A, B, C, D)
fig.show()

## Impulse response

Compute the impulse response with **dss_to_impz** (MIMO).

In [ ]:
ir_len = Fs  # 2 seconds
impulse_response = pyFDN.dss_to_impz(ir_len, delays, A, B, C, D)

## Time domain and play

Plot the impulse response in the time domain and use the audio widget to play it.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display

ir_channel = impulse_response[:, 2, 1]

t = np.arange(ir_len) / Fs
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, ir_channel, color="tab:blue")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Amplitude")
ax.set_title("Homogeneous allpass FDN — impulse response (time domain)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(Audio(ir_channel, rate=Fs))